# Lab 4 – Mini-challenge: the best classifier for a *new* expedition
*Session 5 · about 2 hours + poster*

**The task:** build the classifier with the highest **macro-F1 on the dive-split test set** (79 labels).
You may change the features, the classifier and its settings – anything in this notebook.

**The rules (they are what makes the result honest):**
1. The test set is fixed: `data.split(df, "dive", seed=0)`. Do not change it.
2. **Never tune on the test set.** Try ideas on the *validation* set (a dive split *inside* the training set).
3. When you are done, evaluate **once** on the test set and submit that number – even if it is lower than you hoped.

In [1]:
# Run this cell first. On Google Colab it fetches the course; on your own computer it only loads the data.
import os, subprocess, sys
from pathlib import Path
if "google.colab" in sys.modules:
    if not Path("/content/fathomnetchallange").exists():
        if Path("/content/fathomnetchallange.zip").exists():      # zip from your tutor (drag it into Files)
            subprocess.run(["unzip", "-q", "/content/fathomnetchallange.zip", "-d", "/content"], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/TorbenGl/fathomnetchallange",
                            "/content/fathomnetchallange"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "fathomnet"], check=True)
    os.chdir("/content/fathomnetchallange/course")
sys.path.insert(0, str(Path.cwd()))

from setup_env import setup
ROOT = setup()          # first time: downloads the crops (~15 min). setup(per_category=100) is a quick subset.

import warnings; warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from classifier_lab import data, features, models, evaluate
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})

Dataset ready: /home/user/fathomnetchallange/fathomnet2025_compact  (23698 crop files; exact duplicate boxes are dropped when loading)


In [2]:
df = data.set_target(data.load_dataset(), "label")
train_full, test = data.split(df, "dive", seed=0)          # fixed: the 'new expedition'
train, val = data.split(train_full, "dive", seed=1)        # your playground
idx = lambda part: df.index[df["annotation_id"].isin(part["annotation_id"])]
i_train, i_val, i_full, i_test = idx(train), idx(val), idx(train_full), idx(test)
print(len(i_train), "train /", len(i_val), "validation /", len(i_test), "test crops")

13474 train / 4397 validation / 5793 test crops


## 1. Ideas to try
* other feature sets: `features.get("cnn_resnet50", df)` (2,048 numbers, slower) or `"cnn_efficientnet_b0"`;
  or **combine** feature sets with `np.hstack([X1, X2])`
* other classifier settings: `models.make("logistic_regression", C=1.0)`, `models.make("knn", k=15)`,
  `models.make("mlp", hidden=512)`, `models.make("random_forest", n_trees=500)`
* fewer labels with very few, blurry crops? (no – the test set keeps them; but you can *drop tiny boxes from training*:
  `df.loc[i_train, "box_size"] > 16`)
* **Level 5 (GPU recommended):** fine-tune the whole CNN, see section 3.

In [3]:
X = features.cnn_embedding(df)                              # <- change me
y = df["y"].to_numpy()

def try_on_validation(clf, X):
    res, _ = evaluate.run(clf, X[i_train], y[i_train], X[i_val], y[i_val])
    print(f"validation: accuracy {res['accuracy']:.1%}  macro-F1 {res['macro_f1']:.3f}  ({res['fit_seconds']:.0f}s)")
    return res

for C in [0.01, 0.1, 1.0]:                                  # <- an example of tuning one setting
    print("C =", C, end=": "); try_on_validation(models.make("logistic_regression", C=C), X)

C = 0.01: 

validation: accuracy 42.8%  macro-F1 0.422  (16s)
C = 0.1: 

validation: accuracy 42.4%  macro-F1 0.426  (20s)
C = 1.0: 

validation: accuracy 39.4%  macro-F1 0.400  (22s)


Keep a log of what you tried – it goes on your poster.

In [4]:
log = []   # e.g. log.append({"idea": "logreg C=0.1 on resnet18", "val_macro_f1": 0.52})

## 2. Final evaluation – run once
Train your chosen classifier on **all** training data (train + validation) and evaluate on the test set.

In [5]:
TEAM = "team-name"                                          # <- your team name
FINAL = models.make("logistic_regression", C=0.1)           # <- your final choice
FEATURES = "cnn (resnet18)"                                 # <- describe your features

res, pred = evaluate.run(FINAL, X[i_full], y[i_full], X[i_test], y[i_test])
print(f"TEST: accuracy {res['accuracy']:.1%}   macro-F1 {res['macro_f1']:.3f}")
rep = evaluate.taxonomic_report(y[i_test], pred, data.load_taxonomy())
print("accuracy at phylum level:", f"{rep['accuracy_at_rank']['phylum']:.1%}")

Path("results").mkdir(exist_ok=True)
entry = pd.DataFrame([{"team": TEAM, "features": FEATURES, "classifier": str(FINAL)[:80],
                       "test_accuracy": res["accuracy"], "test_macro_f1": res["macro_f1"],
                       "phylum_accuracy": rep["accuracy_at_rank"]["phylum"]}])
f = Path("results/leaderboard.csv"); entry.to_csv(f, mode="a", header=not f.exists(), index=False)
pd.read_csv(f).sort_values("test_macro_f1", ascending=False)

TEST: accuracy 49.8%   macro-F1 0.475
accuracy at phylum level: 72.6%


,team,features,classifier,test_accuracy,test_macro_f1,phylum_accuracy
0,team-name,cnn (resnet18),"Pipeline(steps=[('standardscaler', StandardSca...",0.497842,0.475432,0.725531


## 3. Level 5 (optional): fine-tune a whole CNN
Up to now the CNN was frozen: we only trained a classifier on its 512 numbers. The competition teams instead
**adjust the CNN itself** to deep-sea animals. On a Colab T4 GPU three epochs should take a few minutes; on a laptop
CPU it takes hours – set `RUN_FINETUNE = True` only on a GPU (*Runtime → Change runtime type → T4 GPU*).

In [6]:
RUN_FINETUNE = False
if RUN_FINETUNE:
    from classifier_lab.finetune import finetune
    pred_ft = finetune(df.loc[i_full], df.loc[i_test], epochs=3)
    print("fine-tuned ResNet-18, TEST:", evaluate.scores(y[i_test], pred_ft))

## 4. Your poster (A3, or 4 slides)
1. **Question & data** – one sentence each; acknowledge FathomNet / MBARI.
2. **Method** – features + classifier in words a biologist understands (no code).
3. **Results** – your test macro-F1 and accuracy, the random-vs-dive comparison (Lab 3), one confusion matrix.
4. **Mistakes** – three example mistakes with crops and a biological explanation.
5. **Verdict** – would you use this classifier for a survey on a new expedition? What would have to be checked by hand?
6. **What you tried** – your log from section 1, including ideas that did not work.